In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import time 

ROOT = Path.cwd().parent          
SRC  = ROOT / "src"
DATA = ROOT / "data"

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print("ROOT:", ROOT)
print("SRC:", SRC)
print("DATA:", DATA)

from portfolio_data_manager import (
    sleep_with_jitter,
    get_aggs_with_backoff,
    read_wide_csv,
    write_wide_csv,
    missing_price_dates,
    build_live_portfolio,
)
from companies import HOLDINGS_INFO  


ROOT: /Users/jamesfourie/Documents/Github/BUFC-fund-dashboard
SRC: /Users/jamesfourie/Documents/Github/BUFC-fund-dashboard/src
DATA: /Users/jamesfourie/Documents/Github/BUFC-fund-dashboard/data


In [2]:
def assert_true(cond, msg):
    if not cond:
        raise AssertionError(msg)

def assert_close(a, b, tol=1e-9, msg="not close"):
    if isinstance(a, (pd.Series, np.ndarray, list)):
        a = np.array(a, dtype=float)
        b = np.array(b, dtype=float)
        if not np.allclose(a, b, atol=tol, rtol=0):
            raise AssertionError(msg)
    else:
        if abs(float(a) - float(b)) > tol:
            raise AssertionError(msg)

print("Assertion helpers ready.")

Assertion helpers ready.


In [3]:
t0 = time.time()
sleep_with_jitter(0.05, jitter=0.0)
elapsed = time.time() - t0

assert_true(elapsed >= 0.05, f"sleep_with_jitter slept too little: {elapsed:.4f}s")
print("sleep_with_jitter OK:", elapsed)


sleep_with_jitter OK: 0.05208921432495117


In [4]:
class FakeAgg:
    def __init__(self, timestamp, close):
        self.timestamp = timestamp
        self.close = close

class FakeClient:
    def __init__(self, fail_times=2):
        self.calls = 0
        self.fail_times = fail_times

    def get_aggs(self, **kwargs):
        self.calls += 1
        if self.calls <= self.fail_times:
            raise Exception("429: rate limited")
        # Return a tiny "aggs" list
        # timestamp in ms for 2025-01-02 and 2025-01-03 UTC
        return [
            FakeAgg(1735776000000, 100.0),  # 2025-01-02
            FakeAgg(1735862400000, 101.0),  # 2025-01-03
        ]

fake = FakeClient(fail_times=2)
aggs = get_aggs_with_backoff(fake, ticker="TEST", from_="2025-01-02", to="2025-01-03")

assert_true(len(aggs) == 2, "get_aggs_with_backoff did not return aggs")
assert_true(fake.calls == 3, f"Expected 3 calls (2 fails + success), got {fake.calls}")
print("get_aggs_with_backoff OK")


Rate-limited; retrying in 1.5s (attempt 1/6)
Rate-limited; retrying in 3.0s (attempt 2/6)
get_aggs_with_backoff OK


In [ ]:
tmp_dir = DATA / "_tmp_tests"
tmp_dir.mkdir(parents=True, exist_ok=True)
tmp_csv = tmp_dir / "roundtrip.csv"

df0 = pd.DataFrame(
    {"date": pd.to_datetime(["2025-01-01", "2025-01-02"]),
     "AAA": [1.0, 2.0],
     "BBB": [np.nan, 5.0]}
).set_index("date")

write_wide_csv(df0, tmp_csv)
df1 = read_wide_csv(tmp_csv)

assert_true(list(df1.columns) == ["AAA", "BBB"], "Columns changed on round-trip")
assert_true(df1.index.min() == pd.Timestamp("2025-01-01"), "Bad index min")
assert_true(df1.index.max() == pd.Timestamp("2025-01-02"), "Bad index max")
assert_close(df1.loc["2025-01-02", "AAA"], 2.0, msg="AAA mismatch")
assert_true(pd.isna(df1.loc["2025-01-01", "BBB"]), "BBB NaN mismatch")

print("read_wide_csv / write_wide_csv OK")

read_wide_csv / write_wide_csv OK


In [ ]:
prices_csv = tmp_dir / "prices.csv"
prices = pd.DataFrame(
    {"date": pd.to_datetime(["2025-01-01", "2025-01-02", "2025-01-03"]),
     "AAA": [10.0, np.nan, 12.0],
     "BBB": [np.nan, np.nan, np.nan]}
)
prices = prices.set_index("date")
write_wide_csv(prices, prices_csv)

miss = missing_price_dates(prices_csv, tickers=["AAA", "BBB", "CCC"], start_date="2025-01-01", end_date="2025-01-03")

assert_true("AAA" in miss, "AAA missing key")
assert_true("BBB" in miss, "BBB missing key")
assert_true("CCC" in miss, "CCC missing key")

assert_true(miss["AAA"] == ["2025-01-02"], f"AAA missing wrong: {miss['AAA']}")
assert_true(miss["BBB"] == ["2025-01-01","2025-01-02","2025-01-03"], f"BBB missing wrong: {miss['BBB']}")
assert_true(miss["CCC"] == ["2025-01-01","2025-01-02","2025-01-03"], f"CCC missing wrong: {miss['CCC']}")

print("missing_price_dates OK")

missing_price_dates OK


In [7]:
snap_csv = tmp_dir / "portfolio_snapshot.csv"
price_csv = tmp_dir / "daily_prices.csv"

# snapshot: 2 days, 2 tickers + CASH
snap = pd.DataFrame(
    {"date": pd.to_datetime(["2025-01-01","2025-01-02"]),
     "CASH": [100.0, 100.0],
     "AAA": [2.0, 2.0],
     "BBB": [1.0, 1.0]}
).set_index("date")
write_wide_csv(snap, snap_csv)

# prices: BBB missing on 1/2 but present on 1/1; should ffill for asof=1/2
prices = pd.DataFrame(
    {"date": pd.to_datetime(["2025-01-01","2025-01-02"]),
     "AAA": [10.0, 12.0],
     "BBB": [20.0, np.nan],
     "CASH": [1.0, 1.0]}
).set_index("date")
write_wide_csv(prices, price_csv)

HINFO = {
    "AAA": {"name": "AAA Inc", "sector": "Tech", "asset_type": "stock", "expected_return": 0.2},
    "BBB": {"name": "BBB Inc", "sector": "Ind",  "asset_type": "stock", "expected_return": 0.1},
    "CASH": {"name": "Cash",   "sector": None,   "asset_type": "cash",  "expected_return": 0.0},
}

df = build_live_portfolio(
    portfolio_snapshot_csv=snap_csv,
    daily_prices_csv=price_csv,
    holdings_info=HINFO,
    asof_date="2025-01-02",
    cash_ticker="CASH",
)

display(df)

# Check price ffill for BBB at 2025-01-02 (should be 20.0)
bbb_price = float(df.loc[df["ticker"]=="BBB", "price"].iloc[0])
assert_close(bbb_price, 20.0, msg="BBB price should have forward-filled to 20")

# Check values
aaa_val = float(df.loc[df["ticker"]=="AAA", "value"].iloc[0])  # 2 * 12 = 24
bbb_val = float(df.loc[df["ticker"]=="BBB", "value"].iloc[0])  # 1 * 20 = 20
cash_val = float(df.loc[df["ticker"]=="CASH", "value"].iloc[0])# 100 * 1 = 100

assert_close(aaa_val, 24.0, msg="AAA value mismatch")
assert_close(bbb_val, 20.0, msg="BBB value mismatch")
assert_close(cash_val, 100.0, msg="CASH value mismatch")

# weights sum to 1 (within floating tolerance)
w_sum = df["weight"].sum()
assert_true(abs(w_sum - 1.0) < 1e-9, f"weights do not sum to 1: {w_sum}")

print("build_live_portfolio OK")


,ticker,name,shares,price,value,sector,asset_type,expected_return,weight
0,CASH,Cash,100.0,1.0,100.0,None,cash,0.0,0.694444
1,AAA,AAA Inc,2.0,12.0,24.0,Tech,stock,0.2,0.166667
2,BBB,BBB Inc,1.0,20.0,20.0,Ind,stock,0.1,0.138889


build_live_portfolio OK


In [ ]:
real_snap = DATA / "portfolio_snapshot.csv"
real_prices = DATA / "daily_prices.csv"

snap = read_wide_csv(real_snap)
assert_true(not snap.empty, "Real portfolio_snapshot.csv is empty")
assert_true(snap.index.is_monotonic_increasing, "Snapshot index not sorted")
assert_true(snap.index.min() <= snap.index.max(), "Snapshot date range invalid")

print("Snapshot OK:", snap.index.min().date(), "→", snap.index.max().date(), "cols:", len(snap.columns))

if real_prices.exists():
    prices = read_wide_csv(real_prices)
    assert_true(prices.index.is_monotonic_increasing, "Prices index not sorted")
    print("Prices OK:", prices.index.min().date(), "→", prices.index.max().date(), "cols:", len(prices.columns))
else:
    print("daily_prices.csv not found yet; skipping real price checks.")

Snapshot OK: 2025-01-01 → 2026-01-23 cols: 26
Prices OK: 2025-01-02 → 2026-01-22 cols: 36
